# QC-C2-03: OpenQASM 3.0 Transpiler across Qiskit, Cirq, CUDA-Q & QSDK
PESU – C-DAC Quantum Hackathon 2026

**Single-file version:** upload only this notebook to Qniverse JupyterLab and run all cells. The first cell unpacks the embedded `qnvkit` package; the second installs target frameworks for verification.

In [ ]:
# Unpacks the embedded qnvkit package next to this notebook (pure Python, needs only numpy).
import base64, io, os, sys, zipfile
_PKG = """
...
ACsCAQAAAA==
"""
_dir = os.path.abspath("qnvkit_pkg")
zipfile.ZipFile(io.BytesIO(base64.b64decode(_PKG.replace("\n", "")))).extractall(_dir)
# Always use THIS embedded copy: put it first and drop any older qnvkit already imported or on the path.
sys.path[:] = [_dir] + [p for p in sys.path if p != _dir]
for _m in [m for m in sys.modules if m == "qnvkit" or m.startswith("qnvkit.")]:
    del sys.modules[_m]
print("qnvkit unpacked to", _dir)

In [ ]:
# Cell: Install execution frameworks for cross-framework verification (Qiskit, Aer, Cirq)
# Purges cache first and uses --no-cache-dir to prevent container disk exhaustion
!pip cache purge
!pip install --no-cache-dir --quiet qiskit qiskit-aer cirq-core

In [ ]:
# Check available target frameworks in this Python environment.
# Note: Code generation works for ALL targets (Qiskit, Cirq, CUDA-Q, QSDK).
# Both Qiskit and Cirq enable cross-framework execution and equivalence checking.
import importlib, importlib.util, sys
importlib.invalidate_caches()
print("Python", sys.version.split()[0])
for _m in ("qiskit", "qiskit_aer", "cirq", "cudaq", "Qniverse", "qniverse"):
    status = "AVAILABLE (simulation & verification active)" if importlib.util.find_spec(_m) is not None else "not installed (code generated & AST verified)"
    print("%-12s %s" % (_m, status))

In [ ]:
import os, sys, json
import qnvkit
from qnvkit.transpile.api import transpile
from qnvkit.qasm.parser3 import parse_qasm3
from qnvkit.qasm.diagnostics import QASMError

print("qnvkit loaded from:", os.path.dirname(qnvkit.__file__))
print("qnvkit transpiler ready!")

In [ ]:
import os, sys
for candidate in ("src", "../src", "CDAC-Hack/src"):
    if os.path.isdir(os.path.join(candidate, "qnvkit")):
        sys.path.insert(0, os.path.abspath(candidate))
        break

import json
from qnvkit.transpile.api import transpile
from qnvkit.qasm.parser3 import parse_qasm3
from qnvkit.qasm.diagnostics import QASMError
print("qnvkit transpiler ready")

## 1. One OpenQASM 3 program: custom gate, modifiers, loop, measurement, condition

In [ ]:
QASM3_SOURCE = """OPENQASM 3.0;
include "stdgates.inc";

gate bell a, b { h a; cx a, b; }      // custom gate
const float theta = pi / 4;

qubit[3] q;
bit[1] m;
bit[2] c;

bell q[0], q[1];
for int i in [0:1] {
    ctrl @ rz(theta * (i + 1)) q[i], q[2];   // modifier -> crz
}
inv @ s q[2];                          // modifier -> sdg
m[0] = measure q[0];
if (m == 1) x q[2];                    // classical condition
c[0] = measure q[1];
c[1] = measure q[2];
"""
print(QASM3_SOURCE)

from qnvkit.viz import draw_circuit, simulate_circuit
print("\n--- Input OpenQASM 3.0 Circuit Diagram ---")
print(draw_circuit(QASM3_SOURCE))


## 2. Generated code, verified (executed) per target

In [ ]:
results = transpile(QASM3_SOURCE, targets=["qiskit", "cirq", "cudaq", "qniverse"], options={"verify": True})
for target, res in results.items():
    print("=" * 70)
    print(target.upper(), "|", res.report.target_version, "| status:", res.report.status)
    print("-" * 70)
    print(res.code)

counts, sim = simulate_circuit(QASM3_SOURCE, shots=1024)
print(f"\n--- Local Simulation of Transpiled Qiskit Circuit ({sim}) ---")
print(f"Measurement Counts: {counts}")


## 3. Structured report (CUDA-Q shown)

In [ ]:
print(results["cudaq"].report.to_json(indent=2))

## 4. Equivalence table
What was executed, fidelity (static circuits) or total variation distance (dynamic circuits, as here),
and what was not run. A framework that is not installed is reported as *not run*, never as verified.

In [ ]:
ver = results["qiskit"].report.metrics["cross_framework_verification"]
print("mode:", ver.get("mode"))
print("%-9s %-9s %-11s %-10s %s" % ("target", "executed", "equivalent", "fid/tvd", "note"))
for name, info in ver["frameworks"].items():
    val = info.get("fidelity", info.get("tvd"))
    print("%-9s %-9s %-11s %-10s %s" % (
        name, info.get("executed"), info.get("equivalent", "-"),
        "-" if val is None else "%.4f" % val, info.get("note") or info.get("error") or ""))

## 4b. Static circuit: statevector fidelity, including the Qniverse QSDK route
For a circuit without mid-circuit measurement or conditions, each generated program is executed and its
statevector compared with the reference simulator. The QSDK program is built with the QSDK, exported with
`qc.qasm()` and compared the same way.

In [ ]:
import qnvkit.qsdk_adapter as adapter
print("QSDK in use:", adapter.qsdk_description())

STATIC = """OPENQASM 3.0;
include "stdgates.inc";
qubit[3] q;
bit[3] c;
h q[0];
cx q[0], q[1];
ctrl @ ry(pi / 3) q[1], q[2];
U(0.3, 0.2, 0.1) q[0];
swap q[0], q[2];
c = measure q;
"""
static_results = transpile(STATIC, targets=["qiskit", "cirq", "cudaq", "qniverse"], options={"verify": True})
ver = static_results["qiskit"].report.metrics["cross_framework_verification"]
print("mode:", ver.get("mode"))
print("%-9s %-9s %-11s %-10s %s" % ("target", "executed", "equivalent", "fid/tvd", "note"))
for name, info in ver["frameworks"].items():
    val = info.get("fidelity", info.get("tvd"))
    print("%-9s %-9s %-11s %-10s %s" % (
        name, info.get("executed"), info.get("equivalent", "-"),
        "-" if val is None else "%.6f" % val, info.get("note") or info.get("error") or ""))
print()
print(static_results["qniverse"].code)

print("\n--- Static Circuit Diagram ---")
print(draw_circuit(STATIC))
counts_static, sim = simulate_circuit(STATIC, shots=1024)
print(f"\n--- Local Simulation of Static Qiskit Circuit ({sim}) ---")
print(f"Measurement Counts: {counts_static}")


## 5. Invalid input: precise `line:column` diagnostics

In [ ]:
bad_programs = {
    "unknown gate": 'OPENQASM 3.0;\nqubit[2] q;\nfoo q[0];\n',
    "index out of range": 'OPENQASM 3.0;\nqubit[2] q;\nh q[5];\n',
}
for title, src in bad_programs.items():
    try:
        parse_qasm3(src)
    except QASMError as e:
        print("[%s] line %d, col %d" % (title, e.line, e.col))
        print(e)
        print()

## 6. An unsupported construct is reported, never emitted as broken code

In [ ]:
UNSUPPORTED = """OPENQASM 3.0;
qubit[2] q;
bit[2] c;
h q[0];
c[0] = measure q[0];
if (c[0] == 1) x q[1];
"""
res = transpile(UNSUPPORTED, targets=["qiskit", "qniverse"])["qniverse"]
print("status:", res.report.status)
print(json.dumps(res.report.unsupported_constructs, indent=2))
compile(res.code, "qniverse", "exec")
print("generated QSDK code is still valid Python and omits the unsupported operation")